<a href="https://colab.research.google.com/github/valerian-drmt/Trading_Projects/blob/main/option_strategy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Import

In [ ]:
!pip install QuantLib
!pip install backtrader
!pip install plotly
!pip install pandasdmx --upgrade
!pip install sdmx1 pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.0/20.0 MB 35.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 5.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.1/154.1 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.2/88.2 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 42.0 MB/s eta 0:00:00
  Attempting uninstall: pydantic
    Found existing installation: pydantic 2.11.7
    Uninstalling pydantic-2.11.7:
      Successfully uninstalled pydantic-2.11.7
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
thinc 8.3.6 requires pydantic<3.0.0,>=2.0.0, but you have pydantic 1.10.22 which is incompatible.
albumentations 2.0.8 requires pydantic>=2.9.2, but you have pydantic 1.10.22 which is incompatible.
langchain 0.3.25 requires pydantic<3.0.0,>=2.7.4, but you have pydant

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import backtrader as bt
import QuantLib as ql
import yfinance as yf
import sdmx
from pandas_datareader import data as web
from datetime import datetime

# Fetch Data

In [ ]:
def get_usd_rate(start_date, end_date):
  # Codes FRED pour les taux souverains US
  tickers = {
      '3M': 'DGS3MO',
      '6M': 'DGS6MO',
      '1Y': 'DGS1',
  }

  # Récupération des données
  df = web.DataReader(list(tickers.values()), 'fred', start_date, end_date)

  # Renommage des colonnes
  df.columns = tickers.keys()
  df = df.dropna()
  return df

In [ ]:
def get_eur_rate(start_date, end_date):
  # Connexion à la BCE
  client = sdmx.Client("ECB")

  # Paramètres de la requête
  keys = {
      'FREQ': 'B',
      'REF_AREA': 'U2',
      'CURRENCY': 'EUR',
      'PROVIDER_FM': '4F',
      'INSTRUMENT_FM': 'G_N_A',
      'DATA_TYPE_FM': ['SR_3M', 'SR_6M', 'SR_1Y']
  }
  params = {'startPeriod': start_date,
            'endPeriod': end_date
            }
  response = client.data(
      resource_id='YC',
      key=keys,
      params= params
  )

  # Conversion en DataFrame
  df = sdmx.to_pandas(response, datetime={"dim": "TIME_PERIOD"})
  # Aplatir les colonnes MultiIndex
  df.columns = df.columns.get_level_values('DATA_TYPE_FM')

  # Renommer les colonnes pour plus de clarté
  df = df.rename(columns={
      'SR_3M': 'EUR_3M',
      'SR_6M': 'EUR_6M',
      'SR_1Y': 'EUR_1Y'
  })
  df = df.dropna()
  return df

In [ ]:
def get_eurusd_price(ticker_fx, start_date, end_date):
  # Download data
  eurusd = yf.download(ticker_fx, start=start_date, end=end_date)
  eurusd = eurusd[['Close']].rename(columns={"Close": "EURUSD"})
  return eurusd

In [ ]:
ticker_fx = "EURUSD=X"
start_date = "2018-01-01"
end_date = "2025-06-13"

In [ ]:
# Fetch data
df1 = get_eur_rate(start_date, end_date)
print("EUR Rates Shape:", df1.shape)
print("EUR Rates Head:\n", df1.head())

df2 = get_usd_rate(start_date, end_date)
print("USD Rates Shape:", df2.shape)
print("USD Rates Head:\n", df2.head())

eurusd = get_eurusd_price(ticker_fx, start_date, end_date)
print("EUR/USD Shape:", eurusd.shape)
print("EUR/USD Head:\n", eurusd.head())

# Combine data
Data = pd.DataFrame(index=eurusd.index)  # Set index directly
Data['EURUSD'] = eurusd['EURUSD']
Data['EUR_Rate_3M'] = df1['EUR_3M']/100
Data['EUR_Rate_6M'] = df1['EUR_6M']/100
Data['EUR_Rate_1Y'] = df1['EUR_1Y']/100
Data['USD_Rate_3M'] = df2['3M']/100
Data['USD_Rate_6M'] = df2['6M']/100
Data['USD_Rate_1Y'] = df2['1Y']/100

# Basic data cleaning
Data = Data.dropna()  # Remove rows with missing values
print("Final Data Shape:", Data.shape)
print("Final Data Head:\n", Data.head())

EUR Rates Shape: (1898, 3)
EUR Rates Head:
 DATA_TYPE_FM    EUR_1Y    EUR_3M    EUR_6M
TIME_PERIOD                               
2018-01-02   -0.712228 -0.745998 -0.738445
2018-01-03   -0.696930 -0.689726 -0.699190
2018-01-04   -0.678375 -0.664711 -0.677090
2018-01-05   -0.675266 -0.659469 -0.672334
2018-01-08   -0.675209 -0.646781 -0.664524


/tmp/ipython-input-24-287999725.py:3: FutureWarning:

YF.download() has changed argument auto_adjust default to True

[*********************100%***********************]  1 of 1 completed

USD Rates Shape: (1863, 3)
USD Rates Head:
               3M    6M    1Y
DATE                        
2018-01-02  1.44  1.61  1.83
2018-01-03  1.41  1.59  1.81
2018-01-04  1.41  1.60  1.82
2018-01-05  1.39  1.58  1.80
2018-01-08  1.45  1.60  1.79
EUR/USD Shape: (1940, 1)
EUR/USD Head:
 Price         EURUSD
Ticker      EURUSD=X
Date                
2018-01-01  1.200495
2018-01-02  1.201158
2018-01-03  1.206345
2018-01-04  1.201043
2018-01-05  1.206884
Final Data Shape: (1832, 7)
Final Data Head:
               EURUSD  EUR_Rate_3M  EUR_Rate_6M  EUR_Rate_1Y  USD_Rate_3M  \
Date                                                                       
2018-01-02  1.201158    -0.007460    -0.007384    -0.007122       0.0144   
2018-01-03  1.206345    -0.006897    -0.006992    -0.006969       0.0141   
2018-01-04  1.201043    -0.006647    -0.006771    -0.006784       0.0141   
2018-01-05  1.206884    -0.006595    -0.006723    -0.006753       0.0139   
2018-01-08  1.203746    -0.006468    -0.006

In [ ]:
# First Chart: EUR OIS Rates
fig1 = go.Figure()
fig1.add_trace(go.Scatter(x=Data.index, y=Data['EUR_Rate_3M'], mode='lines', name='EUR Rate 3M'))
fig1.add_trace(go.Scatter(x=Data.index, y=Data['EUR_Rate_6M'], mode='lines', name='EUR Rate 6M'))
fig1.add_trace(go.Scatter(x=Data.index, y=Data['EUR_Rate_1Y'], mode='lines', name='EUR Rate 1Y'))
fig1.update_layout(
    title='EUR OIS Rates (3M, 6M, 1Y) from ECB',
    xaxis_title='Date',
    yaxis_title='Rate',
    legend_title='Maturity',
    template='plotly_white'
)
fig1.show()

# Second Chart: USD Rates
fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=Data.index, y=Data['USD_Rate_3M'], mode='lines', name='USD Rate 3M'))
fig2.add_trace(go.Scatter(x=Data.index, y=Data['USD_Rate_6M'], mode='lines', name='USD Rate 6M'))
fig2.add_trace(go.Scatter(x=Data.index, y=Data['USD_Rate_1Y'], mode='lines', name='USD Rate 1Y'))
fig2.update_layout(
    title='USD Rates (3M, 6M, 1Y) Over Time',
    xaxis_title='Date',
    yaxis_title='Rate',
    legend_title='Maturity',
    template='plotly_white'
)
fig2.show()

# Third Chart: EUR/USD
fig3 = go.Figure()
fig3.add_trace(go.Scatter(x=Data.index, y=Data['EURUSD'], mode='lines', name='EUR/USD'))
fig3.update_layout(
    title='EUR/USD Exchange Rate',
    xaxis_title='Date',
    yaxis_title='Price',
    legend_title='Exchange Rate',
    template='plotly_white'
)
fig3.show()

# Pricing Options

## European Simple Option

In [ ]:
def european_fx_option_pricing(option_type, S, K, r_usd, r_eur, T, sigma, today):
    """
    Price a European FX option (e.g., EUR/USD) using Black-Scholes-Merton model.

    Parameters:
    - option_type: 'call' or 'put' (call on EUR/USD = right to buy EUR, sell USD)
    - S: Spot exchange rate (e.g., EUR/USD spot rate in USD per EUR)
    - K: Strike exchange rate
    - r_usd: USD interest rate (domestic rate for EUR/USD)
    - r_eur: EUR interest rate (foreign rate for EUR/USD)
    - T: Time to maturity in years
    - sigma: Volatility of the exchange rate
    - today: QuantLib Date object for evaluation date

    Returns:
    - Tuple of (option_price, delta, gamma, theta, vega)
    """
    # Input validation
    if S <= 0 or K <= 0:
        raise ValueError("Spot price (S) and strike price (K) must be positive.")
    if sigma <= 0:
        raise ValueError("Volatility (sigma) must be positive.")
    if T <= 0:
        raise ValueError("Time to maturity (T) must be positive.")

    # 1. Set the evaluation date
    ql.Settings.instance().evaluationDate = today

    # 2. Define the option type
    if option_type.lower() == 'call':
        option_type_ql = ql.Option.Call  # Call on EUR/USD: right to buy EUR
    elif option_type.lower() == 'put':
        option_type_ql = ql.Option.Put  # Put on EUR/USD: right to sell EUR
    else:
        raise ValueError("Invalid option type. Must be 'call' or 'put'.")

    # 3. Create the option object
    payoff = ql.PlainVanillaPayoff(option_type_ql, K)
    # Calculate maturity date (use calendar for accuracy)
    exercise_date = today + ql.Period(int(T * 360), ql.Days)  # Approximate T in days
    exercise = ql.EuropeanExercise(exercise_date)
    european_option = ql.VanillaOption(payoff, exercise)

    # 4. Define the market data
    spot_handle = ql.QuoteHandle(ql.SimpleQuote(S))
    # Domestic rate (USD for EUR/USD)
    rate_handle = ql.YieldTermStructureHandle(ql.FlatForward(today, r_usd, ql.Actual365Fixed()))
    # Foreign rate (EUR for EUR/USD)
    dividend_handle = ql.YieldTermStructureHandle(ql.FlatForward(today, r_eur, ql.Actual365Fixed()))

    vol_handle = ql.BlackVolTermStructureHandle(ql.BlackConstantVol(today, ql.NullCalendar(), sigma, ql.Actual365Fixed()))

    # 5. Create the Black-Scholes process for FX
    bsm_process = ql.BlackScholesMertonProcess(spot_handle, dividend_handle, rate_handle, vol_handle)

    # 6. Create the pricing engine
    engine = ql.AnalyticEuropeanEngine(bsm_process)

    # 7. Set the pricing engine for the option
    european_option.setPricingEngine(engine)

    # 8. Calculate the option price and Greeks
    option_price = european_option.NPV()  # Price in USD per EUR
    delta = european_option.delta()  # Delta w.r.t. spot exchange rate
    gamma = european_option.gamma()  # Gamma w.r.t. spot exchange rate
    theta = european_option.theta() / 365  # Annualized theta (USD per EUR per year)
    vega = european_option.vega() / 100   # Vega per 1% change in volatility

    return option_price, delta, gamma, theta, vega


In [ ]:
T = 3/12

In [ ]:
# Initialize DataFrame to store straddle price and Greeks
Option_Price_List = pd.DataFrame(
    columns=[f"Straddle_{int(T*12)}M_ATMF_USD", "Delta", "Gamma", "Theta", "Vega"],
    index=Data.index
)

vol = Data["EURUSD"].pct_change().dropna().std() * np.sqrt(252)  # Annualized volatility
print(f"Calculated volatility: {vol}")

# Iterate over DataFrame
for index, row in Data.iterrows():
    S = row["EURUSD"]
    K = row["EURUSD"]*np.exp((row["USD_Rate_1Y"]-row["EUR_Rate_1Y"]+1/2*vol**2)*T)
    r = row["USD_Rate_1Y"]
    q = row["EUR_Rate_1Y"]
    sigma = vol
    today = ql.Date(index.day, index.month, index.year)

    try:
        # Price call and put options
        option_price_call_atm, delta_call, gamma_call, theta_call, vega_call = european_fx_option_pricing(
            "call", S, K, r, q, T, sigma, today
        )
        option_price_put_atm, delta_put, gamma_put, theta_put, vega_put = european_fx_option_pricing(
            "put", S, K, r, q, T, sigma, today
        )
        # Calculate straddle price and Greeks
        straddle_price = option_price_call_atm + option_price_put_atm
        straddle_delta = delta_call + delta_put
        straddle_gamma = gamma_call + gamma_put
        straddle_theta = theta_call + theta_put
        straddle_vega = vega_call + vega_put

        # Store in DataFrame
        Option_Price_List.loc[index] = [
            straddle_price,
            straddle_delta,
            straddle_gamma,
            straddle_theta,
            straddle_vega
        ]
    except Exception as e:
        print(f"Error pricing for date {index}: {e}")
        Option_Price_List.loc[index] = [None, None, None, None, None]

Option_Price_List["EURUSD"] = Data["EURUSD"]
Option_Price_List["EUR_Rate_1Y"] = Data["EUR_Rate_1Y"]
Option_Price_List["USD_Rate_1Y"] = Data["USD_Rate_1Y"]
print(Option_Price_List.head())
print(Option_Price_List.columns)

# Créer le graphique avec Plotly
fig = go.Figure()
fig.add_trace(go.Scatter(x=Option_Price_List.index, y=Option_Price_List[f"Straddle_{int(T*12)}M_ATMF_USD"], mode='lines', name='Upper Bound'))

fig.update_layout(
    title="Option Price Over Time",
    xaxis_title="Date",
    yaxis_title="Option Price"
)

fig.show()

Calculated volatility: 0.07544634080618155
           Straddle_3M_ATMF_USD     Delta      Gamma     Theta      Vega  \
Date                                                                       
2018-01-02             0.035988 -0.002065   17.76185 -0.000198  0.004767   
2018-01-03             0.036142  -0.00204  17.684805 -0.000199  0.004788   
2018-01-04             0.035982 -0.002033  17.762076 -0.000198  0.004766   
2018-01-05             0.036157 -0.002016   17.67597 -0.000199   0.00479   
2018-01-08             0.036063 -0.002009  17.722046 -0.000198  0.004777   

              EURUSD  EUR_Rate_1Y  USD_Rate_1Y  
Date                                            
2018-01-02  1.201158    -0.007122       0.0183  
2018-01-03  1.206345    -0.006969       0.0181  
2018-01-04  1.201043    -0.006784       0.0182  
2018-01-05  1.206884    -0.006753       0.0180  
2018-01-08  1.203746    -0.006752       0.0179  
Index(['Straddle_3M_ATMF_USD', 'Delta', 'Gamma', 'Theta', 'Vega', 'EURUSD',
     

## Pivot Point Indicateur